In [1]:
import pandas as pd
import os
import json

In [2]:
def create_dataframe_from_json_folder(folder_path: str) -> pd.DataFrame:
    """
    Reads all JSON files in a given folder, flattens the nested 'averages'
    dictionary, and creates a single pandas DataFrame.

    Args:
        folder_path: The path to the folder containing the JSON files.

    Returns:
        A pandas DataFrame combining the data from all JSON files.
    """
    data_list = []
    
    # Ensure folder_path is a string or Path-like object
    folder_path = str(folder_path)

    print(f"Reading from: {os.path.abspath(folder_path)}")

    # Loop through all files in the given directory
    for filename in os.listdir(folder_path):
        # Check if the file is a JSON file
        if filename.endswith('.json'):
            file_path = os.path.join(folder_path, filename)
            
            try:
                # Open and load the JSON data from the file
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    
                    # Create a dictionary for the row, starting with test_name
                    # Use .get() for safer access in case a key is missing
                    row_data = {
                        'test_name': data.get('test_name', None)
                    }
                    
                    # Get the 'averages' dictionary, defaulting to an empty dict if missing
                    averages = data.get('averages', {})
                    
                    # Update the row_data dictionary with the items from 'averages'
                    # This flattens the nested structure
                    row_data.update(averages)
                    
                    # Add the file's data to our master list
                    data_list.append(row_data)
                    
            except json.JSONDecodeError:
                print(f"Warning: Skipping malformed JSON file: {filename}")
            except Exception as e:
                print(f"Warning: Error reading {filename}: {e}")

    # Create the pandas DataFrame from the list of dictionaries
    if not data_list:
        print("No valid JSON files found to create a DataFrame.")
        return pd.DataFrame()
        
    df = pd.DataFrame(data_list)
    return df

In [ ]:
res = create_dataframe_from_json_folder('../datos/resultados_modelos')

Reading from: /Users/carlosivan/MAIA/Proyecto/finetuning/metricas/resultados_modelos


In [4]:
res.sort_values(by='alignscore')

,test_name,Coleman-Liau,FleschReadingEase,GunningFogIndex,SMOGIndex,Kincaid,DaleChallIndex,alignscore,bertscore_f1
12,llama_v3_beam,13.406573,44.402940,16.934706,14.462083,12.118362,6.861032,0.444208,0.819437
9,gemma3_1b_it,12.126303,55.061812,14.150528,12.575385,10.001917,6.233115,0.457546,0.832400
5,gemma_sinfinetuning,11.354897,67.108953,11.674136,10.807051,7.261822,6.312547,0.466909,0.794859
7,Qwen3-0.6B-Base,10.702674,63.850890,12.753849,11.114326,9.160505,5.259500,0.488796,0.808167
13,llama_v6_beam_sinfinetuning,14.363275,43.512385,16.326798,13.868665,11.760440,7.136427,0.499481,0.820304
4,Qwen3-0.6B-01,14.195165,40.503697,17.661784,14.991514,12.821032,7.377330,0.516178,0.845395
11,gemini-2.5-flash,7.932344,82.479839,6.942596,7.625495,4.193257,4.569995,0.531671,0.826148
10,Qwen3-1.7B-01,14.143647,40.563904,17.544248,14.894495,12.648161,7.339794,0.532564,0.848847
6,gemini-2.5_pro,7.962381,82.612413,6.883697,7.574683,4.211101,4.558748,0.534579,0.826492
0,llama_v6_beam,13.420729,46.720087,15.800495,13.680789,11.623615,6.613555,0.546226,0.829749


In [6]:
res.to_excel('metricas_modelos.xlsx', index=False)